In [ ]:
from finbourne_sdk_utils.jupyter_tools import toggle_code

"""Simple valuation with default recipes

This notebook shows how to value a portfolio using recipes, for an out of the box look at positions and valuations

Attributes
----------
valuation
transactions
recipes
manifests
"""

toggle_code("Hide docstring")

# Simplified Valuation

This notebook illustrates an example of how to use [*GetValuation*](https://www.lusid.com/docs/api/#operation/GetValuation) for a simplified call to the valuation engine that uses a default [*recipe*](https://support.finbourne.com/what-is-a-lusid-recipe-and-how-is-it-used). The default recipe uses a simple `price x quantity` valuation with price source "Lusid".

For an example of a valuation with a customized recipe, see the sample notebook "Valuation with recipe ID".


## Table of contents

- 1. [Load data](#1.-Load-Data)
   * [1.1 Instruments](#1.1-Instruments)
   * [1.2 Portfolio](#1.2-Portfolio)
   * [1.3 Transactions](#1.3-Transactions)
   * [1.4 Quotes](#1.4-Quotes)
- 2. [Run simplified valuations](#2.-Run-simplified-valuations)
    * [2.1 Single-day](#2.1-Single-day)
    * [2.2 Multi-day subtotals](#2.2-Multi-day-subtotals)
    * [2.3 Multi-day ranges](#2.3-Multi-day-ranges)

In [ ]:
# Import system packages

# Import lusid specific packages
# These are the core lusid packages for interacting with the API via Python

import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as models
from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException
from finbourne_sdk_utils.cocoon.cocoon import load_from_data_frame
from finbourne_sdk_utils.cocoon.cocoon_printer import (
    format_instruments_response,
    format_portfolios_response,
    format_transactions_response,
    format_quotes_response,
)
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame

import os
import pandas as pd
from datetime import timedelta
from IPython.core.display import HTML

# Set pandas dataframe display formatting
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format

# Authenticate our user and create our API client
secrets_path = os.getenv("FBN_SECRETS_PATH")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook")

## 1. Load Data

### 1.1 Instruments 

Create a scope and portfolio code, and read the data from the quotes file, containing prices for members of the FTSE100 Index. Here we will simply read the instrument names and identifiers, adding the unique names to LUSID.

In [ ]:
scope = "valuation-simplified"
portfolio_code = "EQUITY_UK"

In [ ]:
instruments_df = pd.read_excel("data/simple-valuation/ftse-100-prices31-Jul-2020-31-Aug-2020.xlsx")[["name", "figi"]].drop_duplicates()
instruments_df.head()

Create a mapping schema for the instruments using the provided FIGIs as the instrument identifiers. The instruments file is loaded into LUSID. 

In [ ]:
instrument_mapping = {
    "identifier_mapping": {
        "Figi": "figi"
    },
    "required": {
        "name": "name"
    },
}

In [ ]:
# Instruments can be loaded using a dataframe with file_type set to "instruments"
result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=instruments_df,
    mapping_required=instrument_mapping["required"],
    mapping_optional={},
    file_type="instruments",
    identifier_mapping=instrument_mapping["identifier_mapping"],
)

succ, failed, errors = format_instruments_response(result)
pd.DataFrame(data=[{"success": len(succ), "failed": len(failed), "errors": len(errors)}])

The instruments should now be viewable in the [LUSID webtool](https://www.lusid.com/app/home) (*Data Management* >>> *Instruments*)

### 1.2 Portfolio

Create a portfolio in LUSID by setting up a mapping schema which can then be used to load the relevant contents.

In [ ]:
portfolio_df = pd.read_excel("data/simple-valuation/portfolio.xlsx")
portfolio_df.head()

In [ ]:
portfolio_mapping = {
    "required": {
        "code": "portfolio_code",
        "display_name": "portfolio_name",
        "base_currency": "$GBP",
    },
    "optional": {"created": "$2020-01-01T00:00:00+00:00"},
}

In [ ]:
# A portfolio can be loaded using a dataframe with file_type = "portfolios"
result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=portfolio_df,
    mapping_required=portfolio_mapping["required"],
    mapping_optional=portfolio_mapping["optional"],
    file_type="portfolios",
    sub_holding_keys=[],
)

succ, failed = format_portfolios_response(result)
pd.DataFrame(data=[{"success": len(succ), "failed": len(failed), "errors": len(errors)}])

### 1.3 Transactions

Create a transaction mapping schema that uses the provided FIGI identifiers to load the data into LUSID.

In [ ]:
transaction_mapping = {
    "identifier_mapping": {
        "Figi": "instrument_id",
    },
    "required": {
        "code": "portfolio_code",
        "transaction_id": "txn_id",
        "type": "txn_type",
        "transaction_price.price": "txn_price",
        "transaction_price.type": "$Price",
        "total_consideration.amount": "txn_consideration",
        "units": "txn_units",
        "transaction_date": "txn_trade_date",
        "total_consideration.currency": "portfolio_base_currency",
        "settlement_date": "txn_settle_date",
    },
    "optional": {},
    "properties": [],
}

In [ ]:
result = load_from_data_frame(
    api_factory=api_factory,
    scope=scope,
    data_frame=portfolio_df,
    mapping_required=transaction_mapping["required"],
    mapping_optional=transaction_mapping["optional"],
    file_type="transactions",
    identifier_mapping=transaction_mapping["identifier_mapping"],
    property_columns=transaction_mapping["properties"],
    properties_scope=scope,
)
    
succ, failed = format_transactions_response(result)
pd.DataFrame(data=[{"success": len(succ), "failed": len(failed), "errors": len(errors)}])

### 1.4 Quotes

Load the source quotes data which containins open and close prices for members of the FTSE100 Index between July 31st and August 31st. This is the pricing data that will be used later in valuation.

In [ ]:
quotes_df = pd.read_excel("data/simple-valuation/ftse-100-prices31-Jul-2020-31-Aug-2020.xlsx")
quotes_df.head()

The current quotes source data uses FIGI as the core unique identifier, but we can call the API and create unique LUSID identifiers (LUID). These will later be used in our valuation call by mapping them against the previously set transactions.

In [ ]:
def add_luid_id(data_frame):
    client_ids = pd.DataFrame(list(data_frame["figi"].unique()), columns=["figi"])

    # Call lusid_instrument_id to the API for creating the LUIDs   
    client_ids["LUID"] = client_ids["figi"].apply(
    lambda x: api_factory.build(lu.InstrumentsApi).get_instrument(
        identifier_type="Figi",
        identifier=x).lusid_instrument_id)
    client_ids = client_ids.set_index("figi")
    data_frame['LUID'] = data_frame["figi"].apply(lambda x: client_ids.loc[x]["LUID"])
    return data_frame

df = add_luid_id(quotes_df)

# Check the first one to see that LUID was added
df[["name", "LUID"]].head(1)

Create a mapping schema for the for the quotes dataframe to read the using the LUIDs.

In [ ]:
quotes_mapping = {
    "quote_id.quote_series_id.instrument_id_type": "$LusidInstrumentId",
    "quote_id.effective_at": "date",
    "quote_id.quote_series_id.provider": "$Lusid",
    "quote_id.quote_series_id.quote_type": "$Price",
    "quote_id.quote_series_id.instrument_id": "LUID",
    "metric_value.unit": "$GBP",
}

We can use the end of day close prices for mapping the quotes source data (pricing data assumed to be using "mid" quotes).

In [ ]:
quotes_mapping["quote_id.quote_series_id.var_field"] ="$mid"
quotes_mapping["metric_value.value"] = "close_price"

result = load_from_data_frame(
    api_factory = api_factory,
    scope=scope,
    data_frame=df,
    mapping_required=quotes_mapping,
    mapping_optional={},
    file_type="quotes"
)

succ, failed, errors = format_quotes_response(result)
display(pd.DataFrame(data=[{"success": len(succ), "failed": len(failed), "errors": len(errors)}]))

## 2. Run simplified valuation

### 2.1 Single-day

Perform a valuation on the portfolio by using the simple valuation call from LUSID. Notice we first create a recipe called "simpleRecipe" to use later in *ValuationRequest*.

- Recipe default attributes:

| Price source/supplier | Instrument ID | Quote Type | Pricing Model | Calculation     |
| :--------------------:| :----------:  | :---------:|:-------------:|:-----------:    |
| LUSID                 | LUID          | Price (mid)| Simple Static | Quanity x price |


In [ ]:
configuration_recipe_api = api_factory.build(lu.ConfigurationRecipeApi)

configuration_recipe = models.ConfigurationRecipe(
        scope=scope,
        code="simpleRecipe",
        market=models.MarketContext(
            market_rules=[
                # define how to resolve the quotes
                models.MarketDataKeyRule(
                    key="Quote.LusidInstrumentId.*",
                    supplier="Lusid",
                    data_scope=scope,
                    quote_type="Price",
                    field="mid",
                ),
            ],
            options=models.MarketOptions(
                default_supplier="Lusid",
                default_instrument_code_type="LusidInstrumentId",
                default_scope=scope,
            ),
        ),
        pricing=models.PricingContext(
            options={"AllowPartiallySuccessfulEvaluation": True},
        ),
    )

upsert_configuration_recipe_response = configuration_recipe_api.upsert_configuration_recipe(
    upsert_recipe_request=models.UpsertRecipeRequest(
        configuration_recipe=configuration_recipe
    )
)

In [ ]:
# Setup the aggregation request 
def aggregation_request(effectiveAt):
    return models.ValuationRequest( 
        recipe_id = models.ResourceId(
            scope = scope,
            code = "simpleRecipe"
        ),
        metrics = [
            models.AggregateSpec(key="Instrument/default/Name", op="Value"),
            models.AggregateSpec(key="Valuation/PV", op="Proportion"),
            models.AggregateSpec(key="Valuation/PV", op="Sum"),
            models.AggregateSpec(key="Holding/default/Units", op="Sum"),
        ],
        group_by=["Instrument/default/Name"],
        # choose the valuation date for the request - set using effectiveAt
        valuation_schedule=models.ValuationSchedule(effective_at=effectiveAt),
        portfolio_entity_ids = [models.PortfolioEntityId(
                                                        scope = scope,
                                                        code = portfolio_code,
                                                        portfolio_entity_type="SinglePortfolio" 
            )]
        )

    

In [ ]:
# Pull the data aggregation by passing the effectiveAt date
aggregation_api = api_factory.build(lu.AggregationApi)
aggregation = aggregation_api.get_valuation(
                                            valuation_request=aggregation_request("2020-08-24T01:01:00.000Z")
            )
pd.DataFrame(aggregation.data)

### 2.2 Multi-day subtotals

Using the same valuation request, we are also able to inspect the evolution of the portfolio holdings and their value for a custom date range. Using a function that groups by <code>"Analytic/default/ValuationDate"</code> the function can call LUSID for an overall PV of the portfolio as a time series.   

In [ ]:
def aggregation_interval_request(effectiveFrom, effectiveAt):
    return models.ValuationRequest( 
        recipe_id = models.ResourceId(
            scope = scope,
            code = "default"
        ),
        metrics = [
            models.AggregateSpec(key="Analytic/default/ValuationDate", op="Value"),
            models.AggregateSpec(key="Valuation/PvInReportCcy", op="Sum"),
        ],
        group_by=["Analytic/default/ValuationDate"],
        # choose the valuation interval for the request - set using effectiveFrom and effectiveAt
        valuation_schedule=models.ValuationSchedule(effective_from = effectiveFrom, effective_at=effectiveAt),
        portfolio_entity_ids = [models.PortfolioEntityId(
                                                        scope = scope,
                                                        code = portfolio_code,
                                                        portfolio_entity_type="SinglePortfolio" 
            )]
        )

In [ ]:
aggregation_api = api_factory.build(lu.AggregationApi)
aggregation = aggregation_api.get_valuation(
                                            valuation_request=aggregation_interval_request(
                                                "2020-08-24T01:01:00.000Z", 
                                                "2020-08-28T01:01:00.000Z")
            )
pd.DataFrame(aggregation.data)

### 2.3 Multi-day ranges

Given the new function now holds data across the selected period, we can also apply other types of specifications in the aggregation metrics. For example, we can see the min/max range for the valuation of each holding in the selected time period. This can illustrate how a stock's volatility can drift the exposure of the portfolio, which can be notable for longer periods. 

In [ ]:
def aggregation_interval_request(effectiveFrom, effectiveAt):
    return models.ValuationRequest( 
        recipe_id = models.ResourceId(
            scope = scope,
            code = "default"
        ),
        metrics = [
            models.AggregateSpec(key="Instrument/default/Name", op="Value"),
            models.AggregateSpec(key="Valuation/PvInReportCcy", op="Min"),
            models.AggregateSpec(key="Valuation/PvInReportCcy", op="Max"),
        ],
        group_by=["Instrument/default/Name"],
        # choose the valuation interval for the request - set using effectiveFrom and effectiveAt
        valuation_schedule=models.ValuationSchedule(effective_from = effectiveFrom, effective_at=effectiveAt),
        portfolio_entity_ids = [models.PortfolioEntityId(
                                                        scope = scope,
                                                        code = portfolio_code,
                                                        portfolio_entity_type="SinglePortfolio" 
            )]
        )

aggregation_api = api_factory.build(lu.AggregationApi)
aggregation = aggregation_api.get_valuation(
                                            valuation_request=aggregation_interval_request(
                                                "2020-08-24T01:01:00.000Z", 
                                                "2020-08-28T01:01:00.000Z")
            )
pd.DataFrame(aggregation.data)

You can also see these valuations in the UI by clicking the below link:

In [ ]:
display(HTML("<h1>Links</h1>"))
display(HTML(f'''
  <a href="https://fbn-ctools.lusid.com/app/dashboard/valuations?scope={scope}&code={portfolio_code}&entityType=Portfolio&recipeScope={scope}&recipeCode=simpleRecipe&effectiveDate=2020-08-24T00:01:00.000Z"
  target="_blank">
    Single day valuation
  </a>'''))